**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 24: Tool Calling 파인튜닝 with Qwen2.5-3B (LoRA)

## 🎯 Tool Calling 파인튜닝 실습

이번 세션에서는 이전에 생성한 Tool Calling 데이터를 사용하여 **Qwen2.5-3B 모델에 Tool Calling 능력을 부여**합니다.

### 실습 흐름

```
데이터 로드 → 포맷팅 → LoRA 설정 → 학습 → Before/After 비교 → 다양한 테스트
```

### 실습 환경

| 항목 | 설정 |
|------|------|
| 모델 | Qwen2.5-3B-Instruct (4bit) |
| 방법 | QLoRA (r=16) |
| 데이터 | Tool Calling 샘플 데이터 |
| GPU | RTX 4060 (8GB VRAM) |

### 학습 목표

- ✅ Tool Calling 데이터를 Chat Template으로 변환
- ✅ QLoRA로 3B 모델 파인튜닝
- ✅ 학습 전후 Tool Calling 능력 비교
- ✅ 다양한 시나리오에서 도구 호출 테스트

## 1️⃣ 환경 설정 및 Qwen2.5-3B 로드 (4bit)

In [1]:
# 필수 라이브러리
import torch
import gc
import os
import json
import time
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 임포트 완료")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 라이브러리 임포트 완료
PyTorch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 2070


In [2]:
# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

print_gpu_memory("시작")

[시작] GPU: 0.0GB / 7.8GB


In [3]:
# 모델 설정
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./output/tool_calling_finetuning"
MAX_SEQ_LENGTH = 1024

# 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 토크나이저 로드
print("⏳ 토크나이저 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ 토크나이저 로드 완료 (vocab: {tokenizer.vocab_size:,})")

⏳ 토크나이저 로딩 중...
✅ 토크나이저 로드 완료 (vocab: 151,643)


In [4]:
# 모델 로드
print("⏳ 모델 로딩 중... (4bit 양자화)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

print(f"✅ 모델 로드 완료")
print(f"📌 모델: {MODEL_NAME}")
print(f"📌 파라미터: {model.num_parameters():,}")
print_gpu_memory("모델 로드 후")

⏳ 모델 로딩 중... (4bit 양자화)


`torch_dtype` is deprecated! Use `dtype` instead!


✅ 모델 로드 완료
📌 모델: Qwen/Qwen2.5-1.5B-Instruct
📌 파라미터: 1,543,714,304
[모델 로드 후] GPU: 1.1GB / 7.8GB


## 2️⃣ Tool Calling 데이터 로드 및 포맷팅

### Tool Calling의 실제 동작 흐름 (2번의 LLM 호출)

```
[1차 LLM 호출]
  입력:  system + user("서울 날씨 알려줘")
  출력:  [TOOL_CALL] [{"name":"get_weather","arguments":{"city":"서울"}}]   ← 학습 대상 ✅
  → LLM 호출 끝. 앱이 출력을 파싱.

[앱이 처리]
  앱이 get_weather(city="서울") 실제 실행
  → {"temperature":15,"condition":"맑음"} 결과 획득                        ← 학습 안 함 ❌

[2차 LLM 호출]
  입력:  이전 대화 전체 + [도구 실행 결과] {"temperature":15,...}
  출력:  "서울은 현재 15도이고 맑습니다."                                   ← 학습 대상 ✅
```

학습 데이터에는 이 **전체 대화가 한 샘플**로 들어가지만,
`DataCollatorForCompletionOnly`가 **assistant 응답(2개)에서만 loss를 계산**합니다.

| 메시지 | Loss 계산 |
|--------|----------|
| system (도구 정의) | ❌ 제외 |
| user (사용자 질문) | ❌ 제외 |
| assistant (도구 호출 JSON) | ✅ **학습** |
| user (도구 실행 결과) | ❌ 제외 |
| assistant (최종 답변) | ✅ **학습** |

In [5]:
# Tool Calling 데이터 로드
data_path = "../data/samples/tool_calling_training_data.json"

with open(data_path, "r", encoding="utf-8") as f:
    tool_data = json.load(f)

print(f"✅ 데이터 로드 완료: {len(tool_data)}개")
print(f"📌 경로: {data_path}")

# 데이터 구조 확인
print(f"\n--- 데이터 예시 ---")
sample = tool_data[0]
for msg in sample["messages"]:
    role = msg["role"]
    content = msg.get("content", "(null)")
    tool_calls = msg.get("tool_calls", None)
    
    if tool_calls:
        for tc in tool_calls:
            func = tc.get("function", {})
            print(f"  [{role}] → {func.get('name')}({func.get('arguments', '')[:50]})")
    else:
        print(f"  [{role}] {str(content)[:80]}")

✅ 데이터 로드 완료: 32개
📌 경로: ../data/samples/tool_calling_training_data.json

--- 데이터 예시 ---
  [system] 당신은 도구를 활용할 수 있는 AI 어시스턴트입니다. 사용 가능한 도구: get_weather(city: str) -> 날씨 정보, calcul
  [user] 서울 날씨 알려줘
  [assistant] → get_weather({"city": "서울"})
  [tool] {"temperature": 15, "condition": "맑음", "humidity": 45}
  [assistant] 서울의 현재 날씨는 기온 15도이고 맑습니다. 습도는 45%입니다.


In [6]:
# Tool Calling 메시지를 학습 텍스트로 변환
def format_tool_calling_to_text(item):
    """
    Tool Calling 대화를 학습용 텍스트로 변환합니다.
    
    변환 규칙:
    - assistant의 tool_calls → [TOOL_CALL] JSON 형식으로 변환 (모델이 생성하도록 학습)
    - tool 응답 → user 역할로 변환 (DataCollator가 loss에서 제외)
    - system, user → 그대로 유지 (DataCollator가 loss에서 제외)
    """
    formatted_messages = []
    
    for msg in item["messages"]:
        role = msg["role"]
        content = msg.get("content", "")
        
        if role == "assistant" and msg.get("tool_calls"):
            # tool_calls → [TOOL_CALL] JSON 형식
            # 추론 시 이 형식을 파싱하여 실제 함수를 호출
            tool_calls = msg["tool_calls"]
            tool_calls_json = json.dumps(
                [{"name": tc["function"]["name"],
                  "arguments": json.loads(tc["function"]["arguments"]) if isinstance(tc["function"]["arguments"], str) else tc["function"]["arguments"]}
                 for tc in tool_calls],
                ensure_ascii=False
            )
            formatted_messages.append({
                "role": "assistant",
                "content": f"[TOOL_CALL] {tool_calls_json}"
            })
        elif role == "tool":
            # tool 응답 → user 역할로 변환 (ChatML이 인식하는 역할)
            # DataCollatorForCompletionOnly가 loss에서 자동 제외
            formatted_messages.append({
                "role": "user",
                "content": f"[도구 실행 결과]\n{content}"
            })
        else:
            formatted_messages.append({
                "role": role,
                "content": content if content else ""
            })
    
    # Qwen2.5 ChatML Template 적용
    try:
        text = tokenizer.apply_chat_template(
            formatted_messages,
            tokenize=False,
            add_generation_prompt=False
        )
        return text
    except Exception as e:
        return None

# 전체 데이터 변환
formatted_texts = []
failed_count = 0

for item in tool_data:
    text = format_tool_calling_to_text(item)
    if text:
        formatted_texts.append(text)
    else:
        failed_count += 1

dataset = Dataset.from_dict({"text": formatted_texts})

print(f"✅ 데이터 포맷팅 완료")
print(f"📌 성공: {len(formatted_texts)}개")
print(f"📌 실패: {failed_count}개")

# 포맷팅 결과 확인 — 어떤 부분이 학습되는지 시각화
print(f"\n--- 포맷팅된 텍스트 예시 ---")
example = dataset[0]["text"]
print(example[:800])
print(f"\n📌 <|im_start|>assistant 뒤의 내용만 loss 계산 대상")
print(f"📌 system, user, [도구 실행 결과]는 loss에서 제외됨")

✅ 데이터 포맷팅 완료
📌 성공: 32개
📌 실패: 0개

--- 포맷팅된 텍스트 예시 ---
<|im_start|>system
당신은 도구를 활용할 수 있는 AI 어시스턴트입니다. 사용 가능한 도구: get_weather(city: str) -> 날씨 정보, calculate(expression: str) -> 계산 결과<|im_end|>
<|im_start|>user
서울 날씨 알려줘<|im_end|>
<|im_start|>assistant
[TOOL_CALL] [{"name": "get_weather", "arguments": {"city": "서울"}}]<|im_end|>
<|im_start|>user
[도구 실행 결과]
{"temperature": 15, "condition": "맑음", "humidity": 45}<|im_end|>
<|im_start|>assistant
서울의 현재 날씨는 기온 15도이고 맑습니다. 습도는 45%입니다.<|im_end|>


📌 <|im_start|>assistant 뒤의 내용만 loss 계산 대상
📌 system, user, [도구 실행 결과]는 loss에서 제외됨


## 3️⃣ LoRA 설정 및 적용

In [7]:
# LoRA 설정
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# LoRA 적용
print("⏳ LoRA 어댑터 적용 중...")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print_gpu_memory("LoRA 적용 후")

⏳ LoRA 어댑터 적용 중...
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
[LoRA 적용 후] GPU: 1.1GB / 7.8GB


## 4️⃣ Trainer로 학습 실행

노트북 19, 20과 동일한 방식:
- `Trainer` + `TrainingArguments` 사용 (SFTTrainer 대신)
- 데이터셋을 직접 토큰화하고 labels에 -100 마스킹 적용
- assistant 응답만 loss 계산

In [8]:
# 학습 전 응답 저장 (Before)
print("="*60)
print("📋 학습 전 Tool Calling 테스트 (Before)")
print("="*60)

SYSTEM_PROMPT = """당신은 도구를 활용할 수 있는 AI 어시스턴트입니다.
사용 가능한 도구:
- get_weather(city: str) -> 날씨 조회
- calculate(expression: str) -> 계산
도구가 필요하면 적절히 호출하세요."""

# 3가지 유형 테스트
TEST_PROMPTS = [
    # 1. 일반 대화 (도구 불필요)
    ("일반 대화", "gradient checkpointing이 뭐야?"),
    ("일반 대화", "안녕하세요 오늘 수업 잘 부탁드립니다"),
    
    # 2. 날씨 도구
    ("날씨 도구", "서울 날씨 알려줘"),
    ("날씨 도구", "부산이랑 제주도 날씨 비교해줘"),
    
    # 3. 계산 도구
    ("계산 도구", "123 곱하기 456은?"),
    ("계산 도구", "원달러 환율 1350원일 때 250달러는 얼마야?"),
]

model.eval()

# 추론을 위해 모든 파라미터를 bfloat16으로 통일
for name, param in model.named_parameters():
    if param.dtype == torch.float32:
        param.data = param.data.to(torch.bfloat16)

before_responses = []
for category, prompt in TEST_PROMPTS:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    before_responses.append(response)
    print(f"\n🔹 [{category}] {prompt}")
    print(f"   {response[:150]}")

print("\n✅ Before 응답 수집 완료")

# Cell 19용 Before 응답도 미리 수집
advanced_prompts = [
    ("일반 대화", "epoch이랑 step 차이가 뭐야?"),
    ("일반 대화", "감사합니다 오늘 많이 배웠어요"),
    ("날씨 도구", "대전 날씨 어때?"),
    ("날씨 도구", "서울이랑 대구 날씨 비교해줘"),
    ("계산 도구", "100 나누기 7은?"),
    ("계산 도구", "15% 할인하면 85000원짜리가 얼마야?"),
    ("도구 조합", "서울 날씨 확인하고 화씨로 변환해줘"),
]

print("\n📋 시나리오 테스트용 Before 응답 수집 중...")
adv_before_responses = []
for category, prompt in advanced_prompts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    adv_before_responses.append(response)
    print(f"  [{category}] {prompt[:40]}...")

print(f"✅ 시나리오 Before 수집 완료: {len(adv_before_responses)}건")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


📋 학습 전 Tool Calling 테스트 (Before)

🔹 [일반 대화] gradient checkpointing이 뭐야?
   Gradient checkpointing는 PyTorch에서 사용되는 기법으로, 모델의 일부 부분을 별도로 저장하고 불러오는 방식을 말합니다. 이 기법은 훈련 과정에서 많은 시간과 메모리 사용량을 줄이는 데 도움이 됩니다.

그럼 어떤 상황에 gradient check

🔹 [일반 대화] 안녕하세요 오늘 수업 잘 부탁드립니다
   네, 안녕하세요! 수업에 대해 준비하고 있을까요? 혹시 어떤 분야의 수업인지 알려주시면 더 도움이 될 것 같습니다.

🔹 [날씨 도구] 서울 날씨 알려줘
   물론이죠, 서울의 현재 날씨 정보를 알려드리겠습니다.

```python
from datetime import datetime

def get_weather(city):
    # 예시로 서울의 날씨 정보를 생성합니다.
    weather_data = {
      

🔹 [날씨 도구] 부산이랑 제주도 날씨 비교해줘
   죄송합니다, 현재 저는 실시간 데이터를 제공하거나 날씨 정보를 제공하는 기능을 가지고 있지 않습니다. 하지만 날씨 정보는 여러 웹사이트나 앱에서 쉽게 확인할 수 있습니다. 부산과 제주도의 날씨를 비교하려면 각 지역의 날씨 API를 사용해야 합니다. 이를 위해 'get_

🔹 [계산 도구] 123 곱하기 456은?
   죄송합니다, 저는 현재의 날씨 정보를 제공하거나 계산을 수행하는 기능이 없습니다. 현재의 날씨 정보는 'get_weather(city: str)' 함수와 계산은 'calculate(expression: str)' 함수로 사용할 수 있습니다. 다른 도움이 필요하시면 알려

🔹 [계산 도구] 원달러 환율 1350원일 때 250달러는 얼마야?
   이 문제를 해결하기 위해 우리는 원달러 환율을 사용해야 합니다. 하지만 현재의 환율 정보는 제공되지 않았습니다. 따라서 이 문제를 직접적으로 해결하는 데 필요한 정보가 부족

In [9]:
# 데이터셋 토큰화 + Labels 마스킹
# assistant 응답만 loss 계산하도록 labels 설정

# assistant 응답 시작/끝 토큰
RESP_TOKEN_IDS = tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
END_TOKEN_ID = tokenizer.encode("<|im_end|>", add_special_tokens=False)[0]

print(f"📌 assistant 시작 토큰: {RESP_TOKEN_IDS}")
print(f"📌 종료 토큰: {END_TOKEN_ID}")

def tokenize_with_labels(example):
    """텍스트를 토큰화하고 assistant 부분만 labels로 설정"""
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    input_ids = tokens["input_ids"]
    
    # 기본: 전부 -100 (loss 계산 안 함)
    labels = [-100] * len(input_ids)
    
    # <|im_start|>assistant\n 위치를 찾아서 그 뒤부터 <|im_end|>까지 labels 설정
    resp_len = len(RESP_TOKEN_IDS)
    j = 0
    while j < len(input_ids) - resp_len:
        if input_ids[j:j+resp_len] == RESP_TOKEN_IDS:
            start = j + resp_len
            end = start
            while end < len(input_ids) and input_ids[end] != END_TOKEN_ID:
                end += 1
            # assistant 응답 + <|im_end|> 토큰까지 학습 대상
            labels[start:end+1] = input_ids[start:end+1]
            j = end + 1
        else:
            j += 1
    
    tokens["labels"] = labels
    return tokens

# 전체 데이터셋에 적용
tokenized_dataset = dataset.map(tokenize_with_labels, remove_columns=["text"])

print(f"\n✅ 토큰화 완료: {len(tokenized_dataset)}개")

# 검증: 첫 번째 샘플
sample = tokenized_dataset[0]
total = len(sample["input_ids"])
train = sum(1 for l in sample["labels"] if l != -100)
print(f"📌 샘플 0: 전체 {total} 토큰, 학습 대상 {train} 토큰 ({train/total*100:.0f}%)")

# 학습 대상 텍스트 확인
train_ids = [l for l in sample["labels"] if l != -100]
print(f"\n--- 학습 대상 텍스트 ---")
print(tokenizer.decode(train_ids)[:300])

📌 assistant 시작 토큰: [151644, 77091, 198]
📌 종료 토큰: 151645


Map: 100%|██████████| 32/32 [00:00<00:00, 1169.62 examples/s]


✅ 토큰화 완료: 32개
📌 샘플 0: 전체 154 토큰, 학습 대상 54 토큰 (35%)

--- 학습 대상 텍스트 ---
[TOOL_CALL] [{"name": "get_weather", "arguments": {"city": "서울"}}]<|im_end|>서울의 현재 날씨는 기온 15도이고 맑습니다. 습도는 45%입니다.<|im_end|>


In [10]:
# Trainer 설정 (노트북 19, 20과 동일 방식)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=20,                   # 소량 데이터 → 충분히 반복
    per_device_train_batch_size=1,         # RTX 4060 안전 설정
    gradient_accumulation_steps=2,
    learning_rate=5e-4,                    # LoRA 표준 lr (노트북 20 참조)
    fp16=False,
    logging_steps=5,
    save_strategy="no",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    report_to="none",
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

# Data Collator: padding + labels 보존
# DataCollatorForLanguageModeling은 labels를 덮어쓰므로 사용 불가
# labels의 -100 마스킹을 유지하면서 padding만 수행
def tool_calling_collator(examples):
    # input_ids, attention_mask padding
    batch = tokenizer.pad(
        [{"input_ids": ex["input_ids"], "attention_mask": ex["attention_mask"]}
         for ex in examples],
        padding=True,
        return_tensors="pt",
    )
    # labels는 -100으로 padding (loss 계산 제외)
    max_len = batch["input_ids"].shape[1]
    padded_labels = []
    for ex in examples:
        lab = ex["labels"]
        padded_labels.append(lab + [-100] * (max_len - len(lab)))
    batch["labels"] = torch.tensor(padded_labels)
    return batch

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=tool_calling_collator,
)

# 학습량 계산
n_samples = len(tokenized_dataset)
steps_per_epoch = n_samples // training_args.gradient_accumulation_steps
total_steps = steps_per_epoch * int(training_args.num_train_epochs)

print(f"📊 학습량 계산")
print(f"📌 데이터: {n_samples}개")
print(f"📌 에포크당 optimizer update: {steps_per_epoch}회")
print(f"📌 총 optimizer update: {total_steps}회")
print(f"\n✅ Trainer 초기화 완료")
print_gpu_memory("학습 시작 전")

/usr/bin/ld: cannot find -lcufile: 그런 파일이나 디렉터리가 없습니다
collect2: error: ld returned 1 exit status


📊 학습량 계산
📌 데이터: 32개
📌 에포크당 optimizer update: 16회
📌 총 optimizer update: 320회

✅ Trainer 초기화 완료
[학습 시작 전] GPU: 1.1GB / 7.8GB


In [11]:
# 학습 실행
print("🚀 Tool Calling 파인튜닝 시작!")
print("="*60)

import time
start_time = time.time()
train_result = trainer.train()
training_time = time.time() - start_time

print("="*60)
print("✅ 학습 완료!")
print(f"📌 소요 시간: {training_time:.1f}초 ({training_time/60:.1f}분)")
print(f"📌 Final Loss: {train_result.training_loss:.4f}")
print(f"📌 Total Steps: {train_result.global_step}")
print_gpu_memory("학습 완료")

🚀 Tool Calling 파인튜닝 시작!


You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
5,1.319100
10,0.621900
15,0.678000
20,0.336500
25,0.252000
30,0.400400
35,0.274800
40,0.193700
45,0.256200
50,0.195900


✅ 학습 완료!
📌 소요 시간: 308.1초 (5.1분)
📌 Final Loss: 0.0979
📌 Total Steps: 320
[학습 완료] GPU: 1.2GB / 7.8GB


## 5️⃣ 학습 전후 Tool Calling 비교

In [12]:
# 학습 후 응답 생성 + 실제 Tool Calling 전체 흐름 시연
print("="*60)
print("📊 학습 전후 Tool Calling 비교 (Before vs After)")
print("="*60)

model.eval()

# 가상 도구 함수 (실제 앱에서는 진짜 API 호출)
import random
def fake_get_weather(city):
    temps = {"서울": 15, "부산": 18, "제주": 20, "대전": 14, "대구": 17}
    conditions = ["맑음", "흐림", "비", "구름 많음"]
    return json.dumps({
        "temperature": temps.get(city, random.randint(10, 25)),
        "condition": random.choice(conditions),
        "humidity": random.randint(30, 80)
    }, ensure_ascii=False)

def fake_calculate(expression):
    try:
        result = eval(expression)
        return json.dumps({"result": result})
    except:
        return json.dumps({"error": "계산 실패"})

FAKE_TOOLS = {
    "get_weather": lambda args: fake_get_weather(args["city"]),
    "calculate": lambda args: fake_calculate(args["expression"]),
}

def generate_response(prompt_messages):
    """1회 LLM 호출"""
    text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def full_tool_calling(prompt):
    """Tool Calling 전체 흐름: 1차 호출 → 도구 실행 → 2차 호출"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    
    # 1차 호출
    response = generate_response(messages)
    
    if not response.startswith("[TOOL_CALL]"):
        # 도구 불필요 → 바로 응답
        return response, None, None
    
    # [TOOL_CALL] 파싱
    try:
        tool_json = response.replace("[TOOL_CALL] ", "")
        tool_calls = json.loads(tool_json)
    except:
        return response, "파싱 실패", None
    
    # 도구 실행
    tool_results = []
    for tc in tool_calls:
        func = FAKE_TOOLS.get(tc["name"])
        if func:
            result = func(tc["arguments"])
            tool_results.append(result)
    
    # 2차 호출: 도구 결과를 전달하여 최종 답변 받기
    messages.append({"role": "assistant", "content": response})
    results_text = "\n".join(tool_results)
    messages.append({"role": "user", "content": f"[도구 실행 결과]\n{results_text}"})
    
    final_response = generate_response(messages)
    return response, tool_results, final_response

# 테스트 실행
for category, prompt in TEST_PROMPTS:
    print(f"\n{'='*60}")
    print(f"🔹 [{category}] {prompt}")
    
    # Before
    before = before_responses[TEST_PROMPTS.index((category, prompt))]
    print(f"  [Before] {before[:120]}")
    
    # After: 전체 Tool Calling 흐름
    tool_call, tool_results, final = full_tool_calling(prompt)
    
    if tool_results is None:
        # 도구 미호출
        print(f"  [After]  {tool_call[:120]}")
        if category == "일반 대화":
            print(f"  ✅ 도구 불필요 → 직접 응답 (정상)")
        else:
            print(f"  ⚠️ 도구 호출 필요하지만 미호출")
    else:
        # 도구 호출 → 전체 흐름
        print(f"  [1차] {tool_call[:120]}")
        print(f"  [도구] {tool_results}")
        print(f"  [최종] {final[:120]}")
        print(f"  ✅ Tool Calling 전체 흐름 완료!")

📊 학습 전후 Tool Calling 비교 (Before vs After)

🔹 [일반 대화] gradient checkpointing이 뭐야?
  [Before] Gradient checkpointing는 PyTorch에서 사용되는 기법으로, 모델의 일부 부분을 별도로 저장하고 불러오는 방식을 말합니다. 이 기법은 훈련 과정에서 많은 시간과 메모리 사용량을 줄이는 데 도움이 
  [After]  gradient checkpointing은 PyTorch에서 지원하는 기능으로, 활성화 함수(activation function)의 복잡한 구조를 checkpointing하여 효율적으로 학습하는 방법입니다.

[TO
  ✅ 도구 불필요 → 직접 응답 (정상)

🔹 [일반 대화] 안녕하세요 오늘 수업 잘 부탁드립니다
  [Before] 네, 안녕하세요! 수업에 대해 준비하고 있을까요? 혹시 어떤 분야의 수업인지 알려주시면 더 도움이 될 것 같습니다.
  [After]  안녕하세요! 오늘 수업이 있다니 기쁩니다. 좋은 하루 되세요!
  ✅ 도구 불필요 → 직접 응답 (정상)

🔹 [날씨 도구] 서울 날씨 알려줘
  [Before] 물론이죠, 서울의 현재 날씨 정보를 알려드리겠습니다.

```python
from datetime import datetime

def get_weather(city):
    # 예시로 서울의 날씨 정보를 생성합니
  [1차] [TOOL_CALL] [{"name": "get_weather", "arguments": {"city": "서울"}}]
  [도구] ['{"temperature": 15, "condition": "맑음", "humidity": 77}']
  [최종] 서울의 현재 날씨는 기온 15도이고 맑습니다. 습도는 77%입니다.
  ✅ Tool Calling 전체 흐름 완료!

🔹 [날씨 도구] 부산이랑 제주도 날씨 비교해줘
  [Before] 죄송합니다, 현재 저는 실시간 데이터를 제공하거나 날씨 정보를 제공하는 기능을 가지고 있

## 6️⃣ 다양한 시나리오 테스트

In [13]:
# 다양한 Tool Calling 시나리오 테스트 (전체 흐름)
print("="*60)
print("📊 다양한 시나리오 테스트 (전체 Tool Calling 흐름)")
print("="*60)

for i, (category, prompt) in enumerate(advanced_prompts):
    print(f"\n{'='*60}")
    print(f"🔹 [{category}] {prompt}")
    
    # Before
    before = adv_before_responses[i]
    print(f"  [Before] {before[:120]}")
    
    # After: 전체 흐름
    tool_call, tool_results, final = full_tool_calling(prompt)
    
    if tool_results is None:
        print(f"  [After]  {tool_call[:120]}")
        if "대화" in category:
            print(f"  ✅ 도구 불필요 → 직접 응답")
        else:
            print(f"  ⚠️ 도구 호출 기대했으나 미호출")
    else:
        print(f"  [1차] {tool_call[:120]}")
        print(f"  [도구] {tool_results}")
        print(f"  [최종] {final[:120]}")
        print(f"  ✅ 전체 흐름 완료!")

# 요약
print(f"\n{'='*60}")
print(f"📊 결과 요약")
print(f"{'='*60}")
print(f"""
💡 Tool Calling 전체 흐름:
  1. user 질문 → LLM 1차 호출
  2. LLM이 [TOOL_CALL] JSON 반환 → 앱이 파싱
  3. 앱이 실제 함수 실행 → 결과 획득
  4. [도구 실행 결과]를 LLM에 2차 전달
  5. LLM이 최종 자연어 답변 생성
""")

📊 다양한 시나리오 테스트 (전체 Tool Calling 흐름)

🔹 [일반 대화] epoch이랑 step 차이가 뭐야?
  [Before] Epoch과 Step는 딥러닝에서 중요한 개념 중 하나로, 각각의 특징을 설명하겠습니다.

1. Epoch (epochs):
   - Epoch은 딥러닝 모델에 한 번의 학습 단계를 의미합니다.
   - 이 단계에서
  [1차] [TOOL_CALL] [{"name": "calculate", "arguments": {"expression": "2 * 3"}}]
  [도구] ['{"result": 6}']
  [최종] epoch과 step의 차이:

- **Epoch**: 전체 데이터셋을 반복하는 횟수. 기준이 1부터 시작하므로, 1 에포크 = 1개的数据.
- **Step**: 현재 로컬에서 활성화된 모델 파라미터 간격. 기준이 
  ✅ 전체 흐름 완료!

🔹 [일반 대화] 감사합니다 오늘 많이 배웠어요
  [Before] 천만에요! 계속 도움이 될 수 있도록 최선을 다하겠습니다. 언제든지 질문해주세요.
  [After]  감사합니다! 도움이 되었다니 기쁩니다. 좋은 하루 되세요!
  ✅ 도구 불필요 → 직접 응답

🔹 [날씨 도구] 대전 날씨 어때?
  [Before] 대전의 현재 날씨는 알려드리기 어렵습니다. 하지만 대전의 날씨 정보를 확인하려면 'get_weather(city: str)' 함수를 사용하여 대전의 날씨를 가져올 수 있습니다. 이 함수에 대전을 전달하면 날씨 정보가
  [1차] [TOOL_CALL] [{"name": "get_weather", "arguments": {"city": "대전"}}]
  [도구] ['{"temperature": 14, "condition": "구름 많음", "humidity": 32}']
  [최종] 현재 대전의 날씨는 기온 14도이고 맑습니다. 습도는 32%입니다.
  ✅ 전체 흐름 완료!

🔹 [날씨 도구] 서울이랑 대구 날씨 비교해줘
  [Before] 물론입니다! 서울과 

In [14]:
# 성능 평가 요약
print("\n📊 Tool Calling 성능 평가")
print("="*60)

print("""
📌 평가 기준:
  1. 도구가 필요한 질문에 도구를 호출했는가?
  2. 올바른 도구를 선택했는가? (날씨 vs 계산)
  3. 올바른 인자를 전달했는가?
  4. 도구가 불필요한 질문에 직접 응답했는가?
  5. 멀티 도구 호출이 필요할 때 여러 도구를 호출했는가?

📌 개선 방향:
  - 더 많은 학습 데이터 (500~1000개)
  - 더 다양한 시나리오 포함
  - 에지 케이스 (모호한 질문, 오타 등)
  - 학습 에포크 증가
  - 더 큰 모델 사용 (7B)
""")


📊 Tool Calling 성능 평가

📌 평가 기준:
  1. 도구가 필요한 질문에 도구를 호출했는가?
  2. 올바른 도구를 선택했는가? (날씨 vs 계산)
  3. 올바른 인자를 전달했는가?
  4. 도구가 불필요한 질문에 직접 응답했는가?
  5. 멀티 도구 호출이 필요할 때 여러 도구를 호출했는가?

📌 개선 방향:
  - 더 많은 학습 데이터 (500~1000개)
  - 더 다양한 시나리오 포함
  - 에지 케이스 (모호한 질문, 오타 등)
  - 학습 에포크 증가
  - 더 큰 모델 사용 (7B)



## 7️⃣ 모델 저장

In [15]:
# LoRA 어댑터 저장
save_path = os.path.join(OUTPUT_DIR, "tool_calling_lora")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ 모델 저장 완료: {save_path}")

# 저장 크기 확인
total_size = sum(
    os.path.getsize(os.path.join(root, f))
    for root, dirs, files in os.walk(save_path)
    for f in files
)

print(f"\n--- 저장된 파일 ---")
for root, dirs, files in os.walk(save_path):
    for f in files:
        fp = os.path.join(root, f)
        size = os.path.getsize(fp)
        print(f"  📄 {f}: {size/1024/1024:.2f} MB")

print(f"\n📌 전체 어댑터 크기: {total_size/1024/1024:.1f} MB")
print(f"📌 이 어댑터를 base 모델에 로드하면 Tool Calling 가능!")

✅ 모델 저장 완료: ./output/tool_calling_finetuning/tool_calling_lora

--- 저장된 파일 ---
  📄 special_tokens_map.json: 0.00 MB
  📄 vocab.json: 2.65 MB
  📄 tokenizer_config.json: 0.00 MB
  📄 adapter_model.safetensors: 35.27 MB
  📄 added_tokens.json: 0.00 MB
  📄 adapter_config.json: 0.00 MB
  📄 README.md: 0.00 MB
  📄 merges.txt: 1.59 MB
  📄 chat_template.jinja: 0.00 MB
  📄 tokenizer.json: 10.89 MB

📌 전체 어댑터 크기: 50.4 MB
📌 이 어댑터를 base 모델에 로드하면 Tool Calling 가능!


In [16]:
# 어댑터 로드 방법 안내
print("\n📋 저장된 어댑터 사용 방법")
print("="*60)

print(f"""
# 어댑터 로드 코드
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Base 모델 로드
base_model = AutoModelForCausalLM.from_pretrained(
    "{MODEL_NAME}",
    quantization_config=bnb_config,
    device_map="auto"
)
base_model.enable_input_require_grads()

# LoRA 어댑터 로드
model = PeftModel.from_pretrained(
    base_model,
    "{save_path}"
)

# 추론
tokenizer = AutoTokenizer.from_pretrained("{save_path}")
""")


📋 저장된 어댑터 사용 방법

# 어댑터 로드 코드
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Base 모델 로드
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)
base_model.enable_input_require_grads()

# LoRA 어댑터 로드
model = PeftModel.from_pretrained(
    base_model,
    "./output/tool_calling_finetuning/tool_calling_lora"
)

# 추론
tokenizer = AutoTokenizer.from_pretrained("./output/tool_calling_finetuning/tool_calling_lora")



In [17]:
# GPU 메모리 정리
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print_gpu_memory("메모리 정리 후")
print("✅ GPU 메모리 정리 완료")

[메모리 정리 후] GPU: 0.0GB / 7.8GB
✅ GPU 메모리 정리 완료


## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

| 항목 | 내용 |
|------|------|
| Tool Calling 데이터 포맷 | assistant의 tool_calls를 `[TOOL_CALL] JSON` 형식으로 변환 |
| Loss Masking | `DataCollatorForCompletionOnly`로 assistant 응답만 학습 |
| QLoRA 파인튜닝 | 1.5B 모델 4bit + LoRA로 RTX 4060에서 학습 |
| 학습 전후 비교 | 도구 호출 능력의 변화 확인 |

### Tool Calling 실제 추론 흐름

```
1. user 질문 → LLM에 전달
2. LLM이 [TOOL_CALL] JSON 반환 → 앱이 파싱
3. 앱이 실제 함수 실행 → 결과 획득
4. [도구 실행 결과]를 LLM에 다시 전달
5. LLM이 최종 자연어 답변 생성
```

### 핵심 포인트

- 🎯 **Loss Masking이 핵심** — 모델은 assistant 응답만 학습, tool 결과는 앱이 제공하는 것
- 🎯 **도구 호출/미호출 균형** — 둘 다 잘 구분해야 합니다
- 🎯 **실무에서는 500+ 데이터** 필요, 지속적 개선 중요
- 🎯 **RTX 4060에서 1.5B QLoRA**로 충분히 학습 가능

### 다음 파트

- ➡️ **Part 4**: 강화학습 (PPO, DPO, GRPO)
- ➡️ 파인튜닝 후 강화학습으로 모델 품질을 더욱 향상시킵니다!